# NetGen Colab Server

Servidor Colab para NetGen basado en `pipeline_v3.ipynb`.

Expone dos rutas:

- `POST /generate`: genera texto con Qwen. Se usa para normalizar el intent y para generar configuraciones.
- `POST /retrieve`: recibe el prompt/query normalizado + variables de acote, filtra el corpus RAG en Colab y devuelve 4 chunks Batfish-compatible.


In [ ]:
!pip -q install fastapi uvicorn pyngrok nest_asyncio sentence-transformers faiss-cpu transformers accelerate bitsandbytes peft pandas pyarrow


In [ ]:
import os
import re
import json
import threading
from typing import Any, Dict, Optional

import faiss
import nest_asyncio
import numpy as np
import pandas as pd
import torch
import uvicorn
from fastapi import FastAPI, HTTPException
from google.colab import drive, userdata
from peft import PeftModel
from pydantic import BaseModel, Field
from pyngrok import ngrok
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig


In [ ]:
# ===== Configuration =====
HF_TOKEN = userdata.get('HF_TOKEN')
NGROK_AUTHTOKEN = userdata.get('NGROK_AUTHTOKEN')

DRIVE_BASE = '/content/drive/MyDrive/dataset eval'
EMB_PATH = DRIVE_BASE + '/rag_embeddings.npy'
META_PATH = DRIVE_BASE + '/rag_metadata.parquet'

BASE_MODEL_PATH = 'Qwen/Qwen2.5-7B-Instruct'
ADAPTER_PATH = 'felbol1/qwen25-7b-networking-v3'
USE_FINETUNED = False

EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
MAX_NEW_TOKENS = 512
MAX_NEW_TOKENS_INTENT = 120
DEFAULT_RETRIEVAL_K = 4

USE_PRODUCT_FILTER = True
USE_BATFISH_COMPAT_FILTER = True

os.environ['HF_TOKEN'] = HF_TOKEN or ''
os.environ['HF_HOME'] = '/root/.cache/huggingface'

if NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

print('Configured NetGen Colab server')
print('Model:', BASE_MODEL_PATH)
print('Embeddings:', EMB_PATH)
print('Metadata:', META_PATH)


In [ ]:
# ===== Drive and artifacts =====
drive.mount('/content/drive', force_remount=True)

for path in [EMB_PATH, META_PATH]:
    if not os.path.exists(path):
        raise FileNotFoundError('File not found: ' + path)
    print('OK', path)

emb_full = np.load(EMB_PATH).astype('float32')
meta_full = pd.read_parquet(META_PATH).reset_index(drop=True)

if emb_full.shape[0] != len(meta_full):
    raise ValueError('Embeddings and metadata are not aligned')

print('Embeddings:', emb_full.shape)
print('Metadata:', meta_full.shape)


In [ ]:
# ===== Batfish-compatible filtering and RAG helpers =====
BATFISH_TEXT_FIELDS = [
    'commands', 'purpose', 'examples', 'section', 'chapter',
    'configuration_guide', 'block_type',
]
BATFISH_SHOW_FIELDS = ['commands']

BATFISH_EXCLUDE_TERMS = [
    'show ip route', 'show bgp summary', 'show interfaces', 'show interface',
    'show arp', 'show logging', 'show processes', 'show cpu', 'show memory',
    'show platform', 'show inventory', 'show environment', 'counter', 'counters',
    'uptime', 'traffic statistics', 'interface statistics', 'log messages',
    'syslog', 'logging host', 'snmp-server', 'trap', 'netflow', 'flow monitor',
    'telemetry', 'gnmi', 'grpc', 'ip sla', 'event manager', 'scheduler',
    'aaa ', 'radius', 'tacacs', 'username ', 'password', 'secret', 'privilege',
    'line vty', 'login local', 'transport input', 'ssh', 'telnet', 'banner',
    'netconf', 'restconf', 'yang', 'guest shell', 'guestshell', 'python script',
    'tcl script', 'api', 'shape average', 'police ', 'priority percent',
    'bandwidth percent', 'queue', 'jitter', 'delay', 'packet loss', 'throughput',
    'dhcp pool', 'ip dhcp', 'dns server', 'domain lookup', 'ntp',
    'ip http server', 'ip http secure-server', 'tftp', 'ftp', 'scp',
    'call-home', 'license', 'smart licensing', 'dmvpn', 'getvpn', 'flexvpn',
    'pki', 'certificate', 'ikev2', 'segment routing', 'srv6', 'pseudowire',
    'vpls', 'l2vpn', 'module', 'transceiver', 'poe', 'stackwise', 'fan',
    'temperature', 'power supply', 'asic',
]

BATFISH_RUNTIME_ONLY_TERMS = [
    'show spanning-tree', 'show mac address-table', 'show etherchannel',
    'show lacp', 'show pagp', 'show ip mroute', 'show ip pim', 'show ip igmp',
    'cam table', 'suspended', 'bpdu guard',
]

def norm(s):
    return ' '.join(str(s).strip().lower().split())

def as_list(value):
    if isinstance(value, (list, tuple, set)):
        return [str(v) for v in value if str(v).strip()]
    if value is None:
        return []
    if isinstance(value, str) and ',' in value:
        return [part.strip() for part in value.split(',') if part.strip()]
    return [str(value)]

def literal_or_match(chunk_value, allowed_values):
    chunk_text = norm(chunk_value)
    return any(norm(allowed) and norm(allowed) in chunk_text for allowed in as_list(allowed_values))

def chunk_matches_arch(row, arch):
    product_ok = True
    if USE_PRODUCT_FILTER:
        product_ok = literal_or_match(row.get('product', ''), arch.get('product', []))
    return (
        literal_or_match(row.get('os', ''), arch.get('os', [])) and
        literal_or_match(row.get('version', ''), arch.get('version', [])) and
        literal_or_match(row.get('device_type', ''), arch.get('device_type', [])) and
        product_ok
    )

def chunk_text(row, fields=BATFISH_TEXT_FIELDS):
    return ' '.join(norm(row.get(field, '')) for field in fields)

def has_disallowed_show(text):
    if 'show ' not in text:
        return False
    return 'show running-config' not in text

def term_in_text(term, text):
    term = norm(term)
    if not term:
        return False
    pattern = r'(?<![a-z0-9])' + re.escape(term) + r'(?![a-z0-9])'
    return re.search(pattern, text) is not None

def batfish_compatible(row):
    show_text = chunk_text(row, fields=BATFISH_SHOW_FIELDS)
    if has_disallowed_show(show_text):
        return False
    full_text = chunk_text(row, fields=BATFISH_TEXT_FIELDS)
    if any(term_in_text(term, full_text) for term in BATFISH_EXCLUDE_TERMS):
        return False
    if any(term_in_text(term, full_text) for term in BATFISH_RUNTIME_ONLY_TERMS):
        return False
    return True

def build_rag_text(row):
    parts = [
        'Device type: ' + str(row.get('device_type', '')),
        'Product: ' + str(row.get('product', '')),
        'OS: ' + str(row.get('os', '')) + '  Version: ' + str(row.get('version', '')),
        'Guide: ' + str(row.get('configuration_guide', '')),
        'Chapter: ' + str(row.get('chapter', '')),
        'Section: ' + str(row.get('section', '')),
        '',
        'Commands:',
        str(row.get('commands', '')),
        '',
        'Examples:',
        str(row.get('examples', '')),
    ]
    return '\n'.join(parts).strip()

def canonical_arch(arch):
    required_keys = ['os', 'version', 'device_type', 'product']
    missing = [key for key in required_keys if key not in arch]
    if missing:
        raise ValueError('Missing arch_base keys: ' + ', '.join(missing))
    return {key: as_list(arch[key]) for key in required_keys}

def arch_cache_key(arch):
    arch = canonical_arch(arch)
    return json.dumps({k: sorted(v) for k, v in arch.items()}, sort_keys=True)

rag_view_cache = {}

def build_rag_view(arch):
    arch = canonical_arch(arch)
    key = arch_cache_key(arch)
    if key in rag_view_cache:
        return rag_view_cache[key]

    mask_arch = meta_full.apply(lambda row: chunk_matches_arch(row, arch), axis=1)
    filtered_df = meta_full[mask_arch]

    if USE_BATFISH_COMPAT_FILTER:
        filtered_df = filtered_df[filtered_df.apply(batfish_compatible, axis=1)]

    selected_pos = filtered_df.index.to_numpy()
    filtered_df = filtered_df.reset_index(drop=True)
    if len(filtered_df) == 0:
        raise ValueError('Architecture/Batfish filter returned zero chunks')

    if 'rag_text' not in filtered_df.columns:
        filtered_df['rag_text'] = filtered_df.apply(build_rag_text, axis=1)

    filtered_emb = emb_full[selected_pos].astype('float32')
    index = faiss.IndexFlatIP(filtered_emb.shape[1])
    index.add(filtered_emb)

    view = {'df': filtered_df, 'emb': filtered_emb, 'index': index, 'arch': arch}
    rag_view_cache[key] = view
    return view


In [ ]:
# ===== Model loading =====
print('Loading embedding model:', EMBEDDING_MODEL_NAME)
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cpu')

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

print('Loading Qwen model:', BASE_MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, token=HF_TOKEN)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    quantization_config=quant_config,
    device_map='auto',
    token=HF_TOKEN,
)

if USE_FINETUNED:
    print('Applying LoRA adapter:', ADAPTER_PATH)
    model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, token=HF_TOKEN)
else:
    model = base_model

model.eval()
print('Model ready')
if torch.cuda.is_available():
    print('VRAM GB:', round(torch.cuda.memory_allocated(0) / 1024**3, 2))


In [ ]:
# ===== Generation and retrieval functions =====
def build_prompt(messages, plain_prompt):
    if getattr(tokenizer, 'chat_template', None):
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return plain_prompt

def generate_text(prompt, max_new_tokens=MAX_NEW_TOKENS):
    messages = [{'role': 'user', 'content': prompt}]
    model_prompt = build_prompt(messages, prompt)
    inputs = tokenizer(model_prompt, return_tensors='pt', truncation=True, max_length=4096).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            temperature=None,
            top_p=None,
            top_k=None,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

def infer_requirement_device_scopes(requirement, arch):
    arch_scopes = [norm(x) for x in as_list(arch.get('device_type', []))]
    text = ' ' + str(requirement).lower() + ' '
    router_patterns = [' r1', ' r2', ' r3', ' r4', 'router1', 'router2', 'router3', 'router4', ' router ', ' routers ']
    switch_patterns = [' sw1', ' sw2', 'switch1', 'switch2', ' switch ', ' switches ']
    has_router = any(p in text for p in router_patterns)
    has_switch = any(p in text for p in switch_patterns)
    scopes = []
    if has_router and 'router' in arch_scopes:
        scopes.append('router')
    if has_switch and 'switch' in arch_scopes:
        scopes.append('switch')
    if not scopes:
        scopes = [s for s in ['router', 'switch'] if s in arch_scopes] or arch_scopes
    return scopes

def row_to_chunk(row, score, retrieval_scope):
    return {
        'score': float(score),
        'section_id': str(row.get('section_id', '')),
        'device_type': str(row.get('device_type', '')),
        'product': str(row.get('product', '')),
        'product_match_kind': str(row.get('product_match_kind', '')),
        'os': str(row.get('os', '')),
        'version': str(row.get('version', '')),
        'configuration_guide': str(row.get('configuration_guide', '')),
        'chapter': str(row.get('chapter', '')),
        'section': str(row.get('section', '')),
        'commands': str(row.get('commands', '')),
        'examples': str(row.get('examples', '')),
        'retrieval_scope': retrieval_scope,
    }

def retrieve_chunks_for_arch(semantic_query, requirement, arch, k=DEFAULT_RETRIEVAL_K):
    view = build_rag_view(arch)
    df = view['df']
    emb = view['emb']
    index = view['index']
    arch = view['arch']

    q_emb = embedding_model.encode([semantic_query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    scopes = infer_requirement_device_scopes(requirement or semantic_query, arch)
    scope_results = []
    seen = set()

    for scope in scopes:
        local_positions = np.flatnonzero(df['device_type'].apply(lambda x: literal_or_match(x, [scope])).to_numpy())
        if len(local_positions) == 0:
            continue
        local_index = faiss.IndexFlatIP(emb.shape[1])
        local_index.add(emb[local_positions])
        scores, indices = local_index.search(q_emb, min(k, local_index.ntotal))
        for score, local_idx in zip(scores[0], indices[0]):
            if local_idx < 0:
                continue
            row = df.iloc[int(local_positions[int(local_idx)])]
            key = str(row.get('section_id', '')) or (str(row.get('configuration_guide', '')), str(row.get('chapter', '')), str(row.get('section', '')), scope)
            if key in seen:
                continue
            scope_results.append(row_to_chunk(row, score, scope))
            seen.add(key)

    if len(scope_results) < k:
        scores, indices = index.search(q_emb, min(k, index.ntotal))
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            row = df.iloc[int(idx)]
            key = str(row.get('section_id', '')) or (str(row.get('configuration_guide', '')), str(row.get('chapter', '')), str(row.get('section', '')), 'architecture_fallback')
            if key in seen:
                continue
            scope_results.append(row_to_chunk(row, score, 'architecture_fallback'))
            seen.add(key)
            if len(scope_results) >= k:
                break

    return sorted(scope_results, key=lambda c: c['score'], reverse=True)[:k]

def build_retrieved_context(chunks):
    if not chunks:
        return 'No relevant Cisco documentation was retrieved.'
    blocks = []
    for i, c in enumerate(chunks, 1):
        parts = [
            '[DOCUMENTATION CHUNK ' + str(i) + ']',
            'Score: ' + '{:.4f}'.format(c.get('score', 0.0)),
            'Retrieval scope: ' + c.get('retrieval_scope', ''),
            'Device type: ' + c.get('device_type', ''),
            'Product: ' + c.get('product', ''),
            'OS: ' + c.get('os', ''),
            'Version: ' + c.get('version', ''),
            'Guide: ' + c.get('configuration_guide', ''),
            'Chapter: ' + c.get('chapter', ''),
            'Section: ' + c.get('section', ''),
            '',
            'Commands:',
            c.get('commands', ''),
            '',
            'Examples:',
            c.get('examples', ''),
        ]
        blocks.append('\n'.join(parts).strip())
    return '\n\n'.join(blocks)


In [ ]:
# ===== FastAPI server =====
class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: Optional[int] = None

class RetrieveRequest(BaseModel):
    semantic_query: str
    requirement: Optional[str] = None
    arch_base: Dict[str, Any]
    k: int = Field(default=DEFAULT_RETRIEVAL_K, ge=1, le=20)

app = FastAPI(title='NetGen Colab API')

@app.post('/generate')
def generate_endpoint(req: GenerateRequest):
    if not req.prompt or not isinstance(req.prompt, str):
        raise HTTPException(status_code=400, detail='prompt is required')
    max_tokens = req.max_new_tokens or MAX_NEW_TOKENS
    text = generate_text(req.prompt, max_new_tokens=max_tokens)
    return {'text': text}

@app.post('/retrieve')
def retrieve_endpoint(req: RetrieveRequest):
    if not req.semantic_query or not isinstance(req.semantic_query, str):
        raise HTTPException(status_code=400, detail='semantic_query is required')
    try:
        arch = canonical_arch(req.arch_base)
    except ValueError as exc:
        raise HTTPException(status_code=400, detail=str(exc))
    try:
        chunks = retrieve_chunks_for_arch(
            semantic_query=req.semantic_query,
            requirement=req.requirement or req.semantic_query,
            arch=arch,
            k=req.k,
        )
    except ValueError as exc:
        raise HTTPException(status_code=404, detail=str(exc))

    return {
        'count': len(chunks),
        'arch_base': arch,
        'chunks': chunks,
        'retrieved_context': build_retrieved_context(chunks),
    }


In [ ]:
# ===== Start ngrok + uvicorn =====
PORT = 8000
nest_asyncio.apply()

try:
    ngrok.kill()
except Exception:
    pass

public_url = ngrok.connect(PORT, 'http').public_url
print('NetGen Colab API URL:', public_url)
print('Generation endpoint:', public_url + '/generate')
print('Retrieval endpoint :', public_url + '/retrieve')

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=PORT, log_level='info')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
